# uv: Fast Python Package Management

## The Core Question

You're setting up a new CFD project. `pip install` takes 3 minutes. Your teammate's environment is broken. The lock file is a mess. How do you get a **fast, reproducible** Python environment?

**uv** is a Python package installer and resolver written in Rust. It's **10-100x faster than pip** and handles virtual environments, locking, and project management in one tool.

## Why uv?

- **Blazing fast**: written in Rust, resolves dependencies in milliseconds (not minutes)
- **All-in-one**: replaces `pip`, `pip-tools`, `virtualenv`, `pyenv`, and `pipx`
- **Reproducible**: `uv.lock` pins exact versions (like `poetry.lock`)
- **Python version management**: `uv python install 3.12` — no pyenv needed

## The Workflow

```
uv init my_cfd_project
        |
        v
  uv add numpy scipy matplotlib
        |
        v
  uv lock  (creates uv.lock with exact versions)
        |
        v
  uv sync  (creates .venv with locked versions)
        |
        v
  uv run python main.py  (runs in the venv)
```

## Setup


In [1]:
import subprocess
import shutil

def check_uv():
    try:
        result = subprocess.run(['uv', '--version'], capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            return result.stdout.strip()
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    return None

uv_version = check_uv()
if uv_version:
    print(f'uv is installed: {uv_version}')
    HAS_UV = True
else:
    print('uv is NOT installed.')
    print('Install: https://docs.astral.sh/uv/getting-started/installation/')
    print('  Windows: powershell -c "irm https://astral.sh/uv/install.ps1 | iex"')
    print('  macOS:   curl -LsSf https://astral.sh/uv/install.sh | sh')
    print()
    print('This notebook will show the commands and expected output.')
    HAS_UV = False

uv is installed: uv 0.13.6 (329bf9448 2026-10-26 x86_64-pc-windows-msvc)


## 1. Initialize a Project

**Concept**: `uv init` creates a minimal project structure: `pyproject.toml`, `.python-version`, and a `main.py`. No boilerplate, no `setup.py`.


In [ ]:
if HAS_UV:
    import os
    import shutil
    test_dir = './uv_demo'
    if os.path.exists(test_dir):
        # Clean up previous demo (ignore errors on Windows where venv files may be locked)
        shutil.rmtree(test_dir, ignore_errors=True)
    
    result = subprocess.run(['uv', 'init', test_dir], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    
    for f in sorted(os.listdir(test_dir)):
        print(f'  {f}')
else:
    print('Expected output of uv init my_cfd_project:')
    print()
    print('  my_cfd_project/')
    print('    pyproject.toml')
    print('    .python-version')
    print('    main.py')
    print('    README.md')


## 2. Add Dependencies

**Concept**: `uv add` adds a dependency to `pyproject.toml` AND resolves it into `uv.lock` in one step. No separate `pip install` needed.


In [ ]:
if HAS_UV:    for dep in ['numpy', 'scipy', 'matplotlib']:        result = subprocess.run(['uv', 'add', dep], capture_output=True, text=True, cwd='./uv_demo')        status = 'OK' if result.returncode == 0 else 'FAIL'        print(f'  uv add {dep:12s} -> {status}')        # Show only the dependencies section of pyproject.toml    with open('./uv_demo/pyproject.toml', 'r') as f:        content = f.read()        print()    print('=== pyproject.toml (dependencies) ===')    lines = content.split('\n')    in_deps = False    for line in lines:        if 'dependencies' in line:            in_deps = True        if in_deps:            print(line)        if in_deps and line.strip() == ']':            break        # Show lock file summary (if it exists)    import os    if os.path.exists('./uv_demo/uv.lock'):        with open('./uv_demo/uv.lock', 'r') as f:            lock_lines = f.readlines()                print()        print(f'=== uv.lock ===')        print(f'Total lines: {len(lock_lines)}')        print('Pinned packages (first 10):')        package_count = 0        for line in lock_lines:            if line.startswith('name = ') and package_count < 10:                print(f'  {line.strip()}')                package_count += 1        print('  ...')        print()        print('The lock file pins exact versions + hashes for reproducibility.')        print('Each package entry includes:')        print('  - Exact version')        print('  - Source URL')        print('  - SHA256 hash (for security)')        print('  - All available wheels (for different platforms)')    else:        print()        print('Note: uv.lock will be created when you run `uv sync` in the next section.')else:    print('Expected output of `uv add numpy scipy matplotlib`:')    print()    print('  uv add numpy     -> OK')    print('  uv add scipy     -> OK')    print('  uv add matplotlib -> OK')    print()    print('=== pyproject.toml (dependencies) ===')    print('dependencies = [')    print('    "matplotlib>=3.10.0",')    print('    "numpy>=2.2.1",')    print('    "scipy>=1.15.0",')    print(']')    print()    print('Note: uv.lock will be created when you run `uv sync` in the next section.')

## 3. Sync and Run

**Concept**: `uv sync` creates a `.venv` with the exact locked versions. `uv run` executes a command inside that venv. No `source .venv/bin/activate` needed.


In [ ]:
if HAS_UV:
    result = subprocess.run(['uv', 'sync'], capture_output=True, text=True, cwd='./uv_demo')
    print(result.stdout)
    
    result = subprocess.run(
        ['uv', 'run', 'python', '-c', 'import numpy; print(f"numpy {numpy.__version__} works")'],
        capture_output=True, text=True, cwd='./uv_demo'
    )
    print(result.stdout)
    
    shutil.rmtree('./uv_demo', ignore_errors=True)
    print('Cleaned up demo project.')
else:
    print('Expected output of `uv sync`:')
    print()
    print('  Using CPython 3.12.4')
    print('  Creating virtual environment at .venv')
    print('  Installed numpy-2.2.1')
    print('  Installed scipy-1.15.0')
    print('  Installed matplotlib-3.10.0')
    print('  3 packages installed in 1.2s')
    print()
    print('Expected output of `uv run python -c "import numpy; print(numpy.__version__)"`:')
    print()
    print('  2.2.1')
    print()
    print('Note: no `source .venv/bin/activate` needed.')
    print('uv run automatically uses the project venv.')

## 4. Why This Matters for Engineering

**Concept**: The #1 problem in scientific computing is **"it works on my machine"**. uv solves this with:

1. **`uv.lock`**: exact versions + hashes. Your teammate runs `uv sync` and gets the SAME environment.
2. **`.python-version`**: pins the Python version. No more "I used 3.10, you used 3.12" bugs.
3. **Speed**: setting up a new environment takes seconds, not minutes. You'll actually do it.


In [ ]:
comparison = (
    "                    pip          uv\n"
    "                  --------     ---------\n"
    "  Resolve deps    30-120s      50-200ms\n"
    "  Install         60-300s      2-10s\n"
    "  Create venv     5-10s        1-2s\n"
    "  Total           2-7 min      3-12s\n"
    "\n"
    "  Lock file       pip-tools    uv.lock (built-in)\n"
    "  Python mgmt     pyenv        uv python (built-in)\n"
    "  Run in venv     source ...   uv run (built-in)\n"
)
print(comparison)
print('For a project you set up once a year, pip is fine.')
print('For a project you iterate on daily, uv saves hours.')

## 5. Revisiting the Setuptools Pipeline with uv

**Concept**: In the setuptools notebook, we built and installed `my_cfd_tool` with:

```
python -m build          # creates dist/*.whl + dist/*.tar.gz
pip install dist/*.whl   # installs the wheel
```

Now let's do the **same pipeline** with `uv`. The package structure and `pyproject.toml` are identical — only the build/install commands change.

### What stays the same
- The `my_cfd_tool/` package (with `__init__.py` and `solver.py`)
- The `pyproject.toml` (with `[build-system]` pointing to setuptools)

### What changes
- `python -m build` → `uv build` (same output, faster)
- `pip install dist/*.whl` → `uv pip install dist/*.whl` (same result, faster)

This is the key insight: **uv is a drop-in replacement for pip**. Your existing setuptools packages work unchanged — you just use `uv` to build and install them faster.


In [ ]:
if HAS_UV:
    import os
    
    # Navigate to the setuptools demo directory (where my_cfd_tool/ and pyproject.toml live)
    pkg_dir = '../01_setuptools'  # relative to this notebook
    
    print('=== Step 1: Build the package with uv ===')
    result = subprocess.run(['uv', 'build'], capture_output=True, text=True, cwd=pkg_dir)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    
    # Show what was built
    dist_dir = os.path.join(pkg_dir, 'dist')
    if os.path.exists(dist_dir):
        print('=== dist/ contents ===')
        for f in sorted(os.listdir(dist_dir)):
            print(f'  {f}')
    
    print()
    print('=== Step 2: Install the wheel with uv ===')
    # Find the wheel file
    wheel = None
    if os.path.exists(dist_dir):
        for f in os.listdir(dist_dir):
            if f.endswith('.whl'):
                wheel = os.path.join(dist_dir, f)
                break
    
    if wheel:
        result = subprocess.run(['uv', 'pip', 'install', wheel], capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
        
        # Verify the install
        result = subprocess.run(
            ['uv', 'run', 'python', '-c', 'from my_cfd_tool import solve; T = solve(1.0, 2.0, 0.0, 100.0, 5); print("Installed and working:", T)'],
            capture_output=True, text=True
        )
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
    else:
        print('No wheel found. Run the build step first.')
else:
    print('Expected output of `uv build` (in the setuptools demo directory):')
    print()
    print('  Building my-cfd-tool 0.1.0')
    print('  Built wheel: dist/my_cfd_tool-0.1.0-py3-none-any.whl')
    print('  Built sdist: dist/my_cfd_tool-0.1.0.tar.gz')
    print()
    print('Expected output of `uv pip install dist/my_cfd_tool-0.1.0-py3-none-any.whl`:')
    print()
    print('  Resolved 1 package in 12ms')
    print('  Installed my-cfd-tool-0.1.0')
    print()
    print('Expected output of `uv run python -c "from my_cfd_tool import solve; ..."`:')
    print()
    print('  Installed and working: [0.0, 25.0, 50.0, 75.0, 100.0]')
    print()
    print('Note: the package was built with setuptools (as declared in pyproject.toml),')
    print('but installed with uv. The build backend and the installer are independent.')

### Key Takeaway

**The build backend and the package installer are separate concerns:**

- **Build backend** (setuptools, hatch, flit, etc.): reads `pyproject.toml` and creates the wheel/sdist. This is declared in `[build-system]`.
- **Package installer** (pip, uv, poetry, etc.): installs the wheel into an environment. This is what you run on the command line.

You can mix and match: build with setuptools, install with uv. Build with hatch, install with pip. The wheel format is the universal interchange.

**For your engineering workflow:**
- Keep using setuptools (or hatch) for building — it's the standard.
- Switch to `uv` for installing — it's faster and handles venvs/locking.
- Your existing packages work unchanged.


## Exercises

1. **Set up a real project**: Create a `uv` project for your CFD post-processing code. Add `numpy`, `scipy`, `matplotlib`, `pandas`. Run `uv sync` and time it.

2. **Reproducibility test**: Commit `pyproject.toml` + `uv.lock` to git. On a different machine, run `uv sync`. Do you get the exact same versions?

3. **Python version**: Use `uv python install 3.11` and set `.python-version` to 3.11. Does your code still work? What breaks?

4. **vs poetry**: Create the same project with both `uv` and `poetry`. Compare: setup time, lock file size, and ease of adding a dependency. Which do you prefer?

5. **CI integration**: Add a GitHub Actions step that uses `uv` instead of `pip`. How much faster is the CI?
